In [8]:
import requests

from spotipy.oauth2 import SpotifyPKCE, CacheFileHandler

from IPython.display import HTML, display

import pandas as pd

## Get Spotify access token

In [ ]:





auth_manager = SpotifyPKCE(
    client_id="f1f9e4ee251c4d0993747e73c48d6cce",
    redirect_uri="http://127.0.0.1:9090/callback",
    scope="playlist-modify-public",
    cache_handler=CacheFileHandler(
        cache_path=".spotify_token_cache"
    )
)

token = auth_manager.get_access_token()

print("succeeded")


Succeed


## Create playlist

In [4]:


headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(
    "https://api.spotify.com/v1/me/playlists",
    headers=headers,
    json={
        "name": "Cross-Genre Music Discovery",
        "description": "My music recommendations",
        "public": True
    },
    timeout=20
)

response.raise_for_status()

playlist_id = response.json()["id"]

print(f"Playlist created : {playlist_id}")

Playlist created : 33aWOeIiteZ2thsep1QGzN


## Empty playlist (only if the playlist already contains tracks)

In [14]:
response = requests.put(
    f"https://api.spotify.com/v1/playlists/{playlist_id}/items",
    headers=headers,
    json={"uris": []}
)

response.raise_for_status()

print("Playlist empty !")

Playlist empty !


## Add tracks to playlist

In [15]:
# Read dataset
df = pd.read_csv("./raw_data/spotify-tracks-dataset-detailed.csv")

# Extract 10 tracks
tracks = df.sample(n=10, random_state=42)


In [16]:


# Extract track ids
track_ids = tracks['track_id']\
    .dropna()\
    .drop_duplicates()\
    .tolist()

# Build tracks URIs
track_uris = [
    f"spotify:track:{track_id}"
    for track_id in track_ids
]

# Add to playlist
response = requests.post(
    f"https://api.spotify.com/v1/playlists/{playlist_id}/items",
    headers=headers,
    json={"uris": track_uris},
    timeout=20
)

response.raise_for_status()


# Show Spotify player

In [17]:


display(HTML(f"""
<iframe
  src="https://open.spotify.com/embed/playlist/{playlist_id}"
  width="100%"
  height="400"
  frameborder="0"
  allow="autoplay; clipboard-write; encrypted-media;
         fullscreen; picture-in-picture"
  loading="lazy">
</iframe>
"""))

# Check track names against dataset

In [18]:
tracks[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
113186,6KwkVtXm8OUp2XffN5k7lY,Hillsong Worship,No Other Name,No Other Name,world-music
42819,2dp5I5MJ8bQQHDoFaNRFtX,Internal Rot,Grieving Birth,Failed Organum,grindcore
59311,5avw06usmFkFrPjX8NxC40,Zhoobin Askarieh;Ali Sasha,Noise A Noise 20.4-1,"Save the Trees, Pt. 1",iranian
91368,75hT0hvlESnDJstem0JgyR,Bryan Adams,All I Want For Christmas Is You,Merry Christmas,rock
61000,4bY2oZGA5Br3pTE1Jd1IfY,Nogizaka46,バレッタ TypeD,月の大きさ,j-idol
96815,2zQt5C0AIv27RhfJCRZdZ4,BaianaSystem,Duas Cidades,"Jah Jah Revolta, Pt. 2",samba
18939,6BctgCJXlgxYeR0ObhLdtR,Joe DeRosa,You Let Me Down,Please Stop Communicating,comedy
72760,1LDQFdGTEXOnycDC8CJ5p1,Cane Hill,A Form of Protest,A Form of Protest,metalcore
25788,2DDR5F7bHFJBiX6lPPsT8O,Kano,I'm Ready (Disco Mix - Original 12 Inch Version),I'm Ready - Radio Edit,disco
87169,2ScU6iEvgb0TIuKiyem9rg,Charlie Brown Jr.,Acústico (Ao Vivo),Proibida Pra Mim (Grazon) - Ao Vivo,r-n-b
